# The Markov Decision Process: the formal model behind all of RL

_Reinforcement Learning_

**States, actions, transition dynamics, reward, and a discount — the five-part contract every reinforcement-learning algorithm is solving.**

The existing ai-mdp lesson introduced the pieces. This lesson states the
       full formal model that every reinforcement-learning algorithm optimizes, and names each
       part precisely &mdash; because RL is notation-heavy and the rest of the curriculum depends on it.

       A Markov Decision Process (MDP) is a 5-tuple
       $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$:
       a set of states $\mathcal{S}$, a set of actions $\mathcal{A}$, the
       transition dynamics $P(s'\mid s,a)$, the reward $R(s,a,s')$, and a
       discount $\gamma$. That is the entire contract.

---

This notebook is a practice scaffold for the **The Markov Decision Process: the formal model behind all of RL** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — numpy

We write the full MDP 5-tuple $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ for a tiny **4-state slippery chain** as plain numpy arrays, then simulate one trajectory under a fixed policy. Seeing each part as a concrete array makes the abstract contract tangible: states and actions are lists, $P$ is a 3-D tensor, $R$ is a table, and $\gamma$ is a scalar.

### Step 1 — Define the states, actions, and transition dynamics $P$

The chain has four states `0,1,2,3` in a line, with state `3` the terminal goal, and two actions (LEFT, RIGHT). The floor is **slippery**: the intended move happens with probability 0.8, otherwise the agent slips and stays put (probability 0.2). We encode this in `P[s, a, s']`, the probability of landing in `s'` after taking action `a` in state `s`, and assert each row is a valid probability distribution.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# A 4-STATE SLIPPERY CHAIN MDP, written as the tuple (S, A, P, R, gamma).
#   States: 0,1,2,3 in a line. State 3 is the terminal GOAL.
#   Actions: 0 = LEFT, 1 = RIGHT. Floor is slippery: intended move
#   happens w.p. 0.8, else the agent STAYS (w.p. 0.2).
S = [0, 1, 2, 3]                 # state space  (script S)
A = [0, 1]                       # action space (script A); 0=LEFT, 1=RIGHT
ACTION_NAME = {0: "LEFT", 1: "RIGHT"}
GOAL, gamma = 3, 0.9
nS, nA = len(S), len(A)

# P[s, a, s'] : probability of next state s' given (s, a).
P = np.zeros((nS, nA, nS))
for s in S:
    left = max(s - 1, 0)         # clamp at the left wall
    right = min(s + 1, nS - 1)   # clamp at the right wall
    P[s, 0, left] += 0.8         # LEFT : 0.8 chance the move happens
    P[s, 0, s] += 0.2            # LEFT : 0.2 chance you slip and stay
    P[s, 1, right] += 0.8        # RIGHT: 0.8 chance the move happens
    P[s, 1, s] += 0.2            # RIGHT: 0.2 chance you slip and stay

# The goal is absorbing (terminal): once there, you stay there.
P[GOAL, :, :] = 0.0
P[GOAL, :, GOAL] = 1.0

# Sanity: for every (s, a) the next-state probs must sum to 1.
assert np.allclose(P.sum(axis=2), 1.0), "transitions must be valid probabilities"


### Step 2 — Define the reward $R$ and inspect the model

The reward is `+1` only for **entering** the goal and `0` otherwise, which we read straight off $P$: the probability of landing in the goal is the expected reward. Once already at the terminal goal the reward is `0`. Printing the RIGHT-action transition matrix and the reward table lets us eyeball the model before we run anything.

In [ ]:
# R[s, a] : expected reward = +1 for ENTERING the goal, else 0.
R = P[:, :, GOAL] * 1.0          # reward only when the move lands in the goal
R[GOAL, :] = 0.0                 # no reward once already terminal

print("Transition tensor P[s, RIGHT, s'] (rows=s, cols=s'):")
print(np.round(P[:, 1, :], 2))
print("Reward table R[s, a]  (cols: LEFT, RIGHT):")
print(np.round(R, 2), "\n")

### Step 3 — Simulate one trajectory under a fixed policy

We fix a deterministic policy `pi(s) = RIGHT` for every state and roll out one episode. At each step we sample the next state from $P(\cdot \mid s, a)$, collect the reward, and accumulate the **discounted** return $G = \sum_t \gamma^t R_t$. Because of slipping, the number of steps to the goal varies run to run; change the seed or the policy to explore that.

In [ ]:
# A deterministic policy: always go RIGHT.
def pi(s):
    return 1                     # always RIGHT

# One environment step: sample s' ~ P(. | s, a) and read off the reward.
def step(s, a):
    s_next = rng.choice(nS, p=P[s, a])
    r = R[s, a]
    return s_next, r

s = 0                            # start in s0
t = 0                            # time step
G = 0.0                          # discounted return
print(f"{'t':>2} {'state':>5} {'action':>6} {'reward':>7}")

while s != GOAL and t < 30:
    a = pi(s)
    s_next, r = step(s, a)
    G += (gamma ** t) * r        # accumulate discounted return G = sum gamma^t R_t
    print(f"{t:>2} {s:>5} {ACTION_NAME[a]:>6} {r:>7.2f}")
    s = s_next
    t = t + 1

print(f"\nReached goal at t={t}.  Discounted return G_0 = {G:.4f}")
# Try changing the seed (rng = default_rng(k)) to see different slippery runs,
# or change pi to 'always LEFT' to watch a bad policy never reach the goal.

## Visualize the data & results

_What does the transition dynamics P(s' | s, a) of the 4-state slippery chain actually look like? We draw the RIGHT-action transition matrix as a heatmap — each row is a current state, each column a next state, the color is the probability — making the 'slip with probability 0.2' structure visible at a glance._

### Step 1 — Rebuild the transition tensor

We reconstruct the same 4-state slippery chain's $P[s, a, s']$ so the visualization cell stands on its own. The construction is identical to the reference implementation: each action moves with probability 0.8 and slips-in-place with probability 0.2, and the goal is absorbing.

In [ ]:
import numpy as np

# Build the same 4-state slippery chain.
nS = 4
GOAL = 3
P = np.zeros((nS, 2, nS))            # P[s, a, s'];  a: 0=LEFT, 1=RIGHT
for s in range(nS):
    left = max(s - 1, 0)
    right = min(s + 1, nS - 1)
    P[s, 0, left] += 0.8            # LEFT  move happens
    P[s, 0, s] += 0.2              # LEFT  slip-stay
    P[s, 1, right] += 0.8           # RIGHT move happens
    P[s, 1, s] += 0.2              # RIGHT slip-stay

# Absorbing goal.
P[GOAL, :, :] = 0.0
P[GOAL, :, GOAL] = 1.0

### Step 2 — Read off the RIGHT-action matrix

Slicing out `P[:, 1, :]` gives the 4x4 matrix of next-state probabilities under the RIGHT action — exactly the grid that gets drawn as the heatmap. Printing it (and confirming each row sums to 1) shows the diagonal 0.2 slip-stay band plus the 0.8 move-right band before we color it in.

In [ ]:
P_right = P[:, 1, :]                 # the RIGHT-action transition matrix
print(np.round(P_right, 2))          # -> the 4x4 grid plotted as the heatmap

# Each row must sum to 1.0 -> a valid probability distribution.
print("row sums:", P_right.sum(axis=1))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** An agent's state is the current screen pixels of a Pong game. Is this state Markov? If not, how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what the optimal action depends on. — _In Pong you need the ball's direction and speed to decide where to move the paddle._
- Check whether one frame reveals that. — _A single still frame shows the ball's position but NOT its velocity — you cannot tell which way it is moving._
- Conclude and fix. — _One frame is non-Markov. Stack the last few frames into the state so velocity is recoverable — this is exactly what the DQN (Deep Q-Network) Atari work did._

**Answer:** No &mdash; a single frame is not Markov because it hides velocity. Fix: make the state a stack of the last 2&ndash;4 frames so the dynamics depend only on the (enlarged) current state again.

</details>

**Problem 2.** In the 4-state chain ($\gamma = 0.9$), a run reaches the goal $s_3$ in exactly 4 steps, earning $+1$ only on the final step. What is the return $G_0$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate where the reward lands. — _Reward $+1$ is received entering the goal, on step 4 — i.e. as $R_3$ (the reward at $t=3$, since the first reward is $R_0$)._
- Apply the discount $\gamma^k$. — _A reward at offset $k$ from the start counts for $\gamma^k$. Here $k = 3$, so the factor is $\gamma^3 = 0.9^3$._
- Compute. — _$0.9^3 = 0.729$. All earlier rewards are $0$, so they add nothing._

**Answer:** $G_0 = \gamma^3 \cdot 1 = 0.9^3 = 0.729$. The extra step shrinks the reward versus the 3-step run's $0.81$ &mdash; discounting rewards speed.

</details>

**Problem 3.** Why does discounting with $\gamma \lt 1$ guarantee the infinite-horizon return is a finite number?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bound each reward. — _Assume rewards never exceed some $R_{\max}$ in size, $|R_t| \le R_{\max}$._
- Bound the whole sum. — _$|G_t| \le \sum_{k\ge0}\gamma^k R_{\max} = R_{\max}\sum_{k\ge0}\gamma^k$, a geometric series._
- Sum the geometric series. — _For $0\le\gamma\lt1$, $\sum_{k\ge0}\gamma^k = 1/(1-\gamma)$, which is finite._

**Answer:** Because $|G_t| \le R_{\max}/(1-\gamma)$, a finite bound. With $\gamma = 1$ the geometric series diverges and the return can be infinite, which is why infinite-horizon RL discounts (or uses average reward / episodic termination instead).

</details>